In [2]:
import math
import torch
import torch.nn as nn
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader,Dataset
from torch.nn import TransformerDecoder,TransformerDecoderLayer,TransformerEncoder,TransformerEncoderLayer
import io
import time
import pandas as pd
import re
import sentencepiece as spm


In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())
device = torch.device('cuda')

True
Tesla T4


In [4]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/hitenm/kftt-dataset/test_kftt.parquet
/kaggle/input/datasets/hitenm/kftt-dataset/train_kftt.parquet
/kaggle/input/datasets/hitenm/kftt-dataset/validation_kftt.parquet


In [5]:
train_kftt = pd.read_parquet('/kaggle/input/datasets/hitenm/kftt-dataset/train_kftt.parquet')
train_kftt

,translation
0,"{'en': 'Known as Sesshu (1420 - 1506), he was ..."
1,{'en': 'He revolutionized the Japanese ink pai...
2,"{'en': 'He was given the posthumous name ""Toyo..."
3,"{'en': 'Born in Bicchu Province, he moved to S..."
4,{'en': 'Later he accompanied a mission to Ming...
...,...
440284,{'en': '1. the highest government post for the...
440285,{'en': '2. the government post for the court l...
440286,{'en': 'Naishi no kami attended on the emperor...
440287,"{'en': 'Later, Naishi no kami was ranked with ..."


In [6]:
test_kftt = pd.read_parquet('/kaggle/input/datasets/hitenm/kftt-dataset/test_kftt.parquet')
test_kftt

,translation
0,"{'en': 'Infobox Buddhist', 'ja': 'InfoboxBuddh..."
1,{'en': 'Dogen was a Zen monk in the early Kama...
2,"{'en': 'The founder of Soto Zen', 'ja': '曹洞宗の開..."
3,{'en': 'Later in his life he also went by the ...
4,{'en': 'Within the sect he is referred to by t...
...,...
1156,"{'en': 'Shugo Daimyo, Sengoku Daimyo', 'ja': '..."
1157,"{'en': 'Shinpan, Fudai Daimyo, Tozama Daimyo',..."
1158,"{'en': 'Domains', 'ja': '藩'}"
1159,"{'en': 'Lords', 'ja': '諸侯'}"


In [7]:
val_kftt = pd.read_parquet('/kaggle/input/datasets/hitenm/kftt-dataset/validation_kftt.parquet')
val_kftt

,translation
0,{'en': 'Rinzai Zen Buddhism is one of the Chin...
1,"{'en': 'He was known as ""RINZAI of Katu (meani..."
2,{'en': 'With its Zen Talks that try to awaken ...
3,"{'en': 'Rinzai School in China', 'ja': '中国における..."
4,"{'en': 'As the name implies, Rinzai School sta..."
...,...
1162,"{'en': 'Kasho: Kegon sect, Tendai sect, etc.',..."
1163,"{'en': 'Osho: Zen sects, Jodo sect, etc.', 'ja..."
1164,"{'en': 'Waja', 'ja': '和闍(わじゃ)'}"
1165,"{'en': '* In broadcasting, the pronunciation o..."


In [8]:
len(train_kftt),len(test_kftt),len(val_kftt)

(440289, 1161, 1167)

In [9]:
print(train_kftt.info())
print('---------------------------------------------------')
print(test_kftt.info())
print('---------------------------------------------------')
print(val_kftt.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440289 entries, 0 to 440288
Data columns (total 1 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   translation  440289 non-null  object
dtypes: object(1)
memory usage: 3.4+ MB
None
---------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1161 entries, 0 to 1160
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   translation  1161 non-null   object
dtypes: object(1)
memory usage: 9.2+ KB
None
---------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1167 entries, 0 to 1166
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   translation  1167 non-null   object
dtypes: object(1)
memory usage: 9.2+ KB
None


In [10]:
print(train_kftt['translation'])
print('---------------------------------------------------')

print(test_kftt['translation'])
print('---------------------------------------------------')

print(val_kftt['translation'])
print('---------------------------------------------------')

0         {'en': 'Known as Sesshu (1420 - 1506), he was ...
1         {'en': 'He revolutionized the Japanese ink pai...
2         {'en': 'He was given the posthumous name "Toyo...
3         {'en': 'Born in Bicchu Province, he moved to S...
4         {'en': 'Later he accompanied a mission to Ming...
                                ...                        
440284    {'en': '1. the highest government post for the...
440285    {'en': '2. the government post for the court l...
440286    {'en': 'Naishi no kami attended on the emperor...
440287    {'en': 'Later, Naishi no kami was ranked with ...
440288                                 {'en': '', 'ja': ''}
Name: translation, Length: 440289, dtype: object
---------------------------------------------------
0       {'en': 'Infobox Buddhist', 'ja': 'InfoboxBuddh...
1       {'en': 'Dogen was a Zen monk in the early Kama...
2       {'en': 'The founder of Soto Zen', 'ja': '曹洞宗の開...
3       {'en': 'Later in his life he also went by the ...
4      

In [11]:
print(train_kftt['translation'][0])
print('---------------------------------------------------')

print(test_kftt['translation'][56])
print('---------------------------------------------------')

print(val_kftt['translation'][0])
print('---------------------------------------------------')

{'en': 'Known as Sesshu (1420 - 1506), he was an ink painter and Zen monk active in the Muromachi period in the latter half of the 15th century, and was called a master painter.', 'ja': '雪舟（せっしゅう、1420年（応永27年）-1506年（永正3年））は号で、15世紀後半室町時代に活躍した水墨画家・禅僧で、画聖とも称えられる。'}
---------------------------------------------------
{'en': '"Shobogenzo Zuimonki" Translated by Yahoko MIZUNO. Chikuma Scholastic Collection, 1992.', 'ja': '『正法眼蔵随聞記』（水野弥穂子訳.ちくま学芸文庫,1992年）'}
---------------------------------------------------
{'en': 'Rinzai Zen Buddhism is one of the Chinese five Houses/seven Schools of Zen (Rinzai, Igyou, Soto, Unmon, Hogan) and Gigen RINZAI (? - 867) of Tang was its founder.', 'ja': '臨済宗（臨濟宗、りんざいしゅう）は、中国禅五家七宗（ごけしちしゅう）（臨済、潙仰宗、曹洞宗、雲門宗、法眼宗）のひとつで、唐の臨済義玄（？-867年）を宗祖とする。'}
---------------------------------------------------


In [12]:
print(test_kftt['translation'][56]['en'])
print('--------------------------------------------------')
print(test_kftt['translation'][56]['ja'])

"Shobogenzo Zuimonki" Translated by Yahoko MIZUNO. Chikuma Scholastic Collection, 1992.
--------------------------------------------------
『正法眼蔵随聞記』（水野弥穂子訳.ちくま学芸文庫,1992年）


In [13]:
max_len = 0
for pair in train_kftt['translation']:
    max_len = max(len(pair['en']),max_len)
max_len    
    

2037

In [14]:
def clean(text):
    return " ".join(text.split())

kept = 0
skipped = 0

with open('train.en','w',encoding = 'utf-8') as f_en,open('train.ja','w',encoding = 'utf-8') as f_ja:
         

         for pair in train_kftt['translation']:
             
             en = clean(pair['en'])
             ja = clean(pair['ja'])
             
             if not en or not ja:
                skipped += 1
                continue
                 
             f_en.write(en + "\n")
             f_ja.write(ja + "\n")
             kept += 1

print("kept:", kept, "skipped:", skipped)


kept: 440286 skipped: 3


In [15]:
with open('train.en', encoding='utf-8') as f:
    en_lines = f.readlines()

with open('train.ja', encoding='utf-8') as f:
    ja_lines = f.readlines()

print("English lines:", len(en_lines))
print("Japanese lines:", len(ja_lines))
assert len(en_lines) == len(ja_lines)

English lines: 440286
Japanese lines: 440286


In [17]:
spm.SentencePieceTrainer.train(
    input = 'train.en',
    model_prefix = 'en',
    vocab_size = 16000,
    character_coverage=1.0,
    byte_fallback=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
)

spm.SentencePieceTrainer.train(
    input = 'train.ja',
    model_prefix = 'ja',
    vocab_size = 16000,
    character_coverage=0.9999,
    byte_fallback=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
)

I0000 00:00:1794983818.173330      49 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: train.en
  input_format: 
  model_prefix: en
  model_type: UNIGRAM
  vocab_size: 16000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_priv

True

In [18]:
eng = spm.SentencePieceProcessor(model_file = 'en.model')

text = "It took me a long time to find the motivation to write a new article. There were many things I wanted to write about but I didn’t have much time for it after having given birth to my son."
pieces = eng.encode(text,return_type = str)
ids = eng.encode(text,return_type = int)

print(f"Pieces: {pieces}")
print(f"IDs:    {ids}")

print(eng.decode(ids))
print(eng.decode(pieces))

Pieces: ['▁It', '▁took', '▁me', '▁a', '▁long', '▁time', '▁to', '▁find', '▁the', '▁mo', 'ti', 'vation', '▁to', '▁write', '▁a', '▁new', '▁article', '.', '▁There', '▁were', '▁many', '▁things', '▁I', '▁wanted', '▁to', '▁write', '▁about', '▁but', '▁I', '▁did', 'n', '’', 't', '▁have', '▁much', '▁time', '▁for', '▁it', '▁after', '▁having', '▁given', '▁birth', '▁to', '▁my', '▁son', '.']
IDs:    [313, 527, 1518, 269, 631, 351, 268, 4048, 260, 1592, 1045, 6331, 268, 4055, 269, 500, 1622, 263, 471, 290, 401, 1951, 402, 3234, 268, 4055, 407, 314, 402, 536, 350, 13045, 344, 345, 1441, 351, 279, 289, 328, 811, 511, 738, 268, 1796, 359, 263]
It took me a long time to find the motivation to write a new article. There were many things I wanted to write about but I didn’t have much time for it after having given birth to my son.
It took me a long time to find the motivation to write a new article. There were many things I wanted to write about but I didn’t have much time for it after having given birth t

In [19]:
ja = spm.SentencePieceProcessor(model_file = 'ja.model')

text = "新しい記事を書こうという気持ちになるまで長い時間がかかった。書きたいことはたくさんあったけれど、息子を産んだ後は書く時間があまりなかった。"
pieces = ja.encode(text,return_type = str)
ids = ja.encode(text,return_type = int)

print(f"Pieces: {pieces}")
print(f"IDs:    {ids}")

print(ja.decode(ids))
print(ja.decode(pieces))

Pieces: ['▁', '新しい', '記事', 'を', '書', 'こう', 'という', '気', '持ち', 'になる', 'まで', '長い', '時間', 'が', 'かかった', '。', '書き', 'たい', 'ことは', 'たく', 'さん', 'あった', 'け', 'れ', 'ど', '、', '息子', 'を産んだ', '後は', '書', 'く', '時間', 'が', 'あまり', 'なかった', '。']
IDs:    [263, 2458, 3423, 270, 367, 989, 298, 669, 1535, 845, 409, 2743, 1402, 267, 9619, 261, 1552, 1383, 1343, 5301, 1219, 2254, 768, 617, 974, 260, 2860, 12104, 1994, 367, 329, 1402, 267, 2395, 749, 261]
新しい記事を書こうという気持ちになるまで長い時間がかかった。書きたいことはたくさんあったけれど、息子を産んだ後は書く時間があまりなかった。
新しい記事を書こうという気持ちになるまで長い時間がかかった。書きたいことはたくさんあったけれど、息子を産んだ後は書く時間があまりなかった。


In [20]:
ja_train = [p["ja"] for p in train_kftt["translation"]]
en_train = [p["en"] for p in train_kftt["translation"]]


sp_ja = spm.SentencePieceProcessor(model_file="ja.model")
sp_en = spm.SentencePieceProcessor(model_file="en.model")



In [21]:
src = sp_ja.encode(ja_train, out_type=int, add_eos=True)
tgt = sp_en.encode(en_train, out_type=int, add_bos=True, add_eos=True)

import numpy as np
sl = np.array([len(x) for x in src])
tl = np.array([len(x) for x in tgt])
for limit in (64, 100, 128, 200):
    keep = (sl <= limit) & (tl <= limit)
    print(limit, "keeps", keep.mean() * 100, "percent")

64 keeps 91.02839271478507 percent
100 keeps 98.45283438832222 percent
128 keeps 99.54121043223883 percent
200 keeps 99.95480241386907 percent


In [22]:
LIMIT = 128
keep = (sl <= LIMIT) & (tl <= LIMIT)      # sl, tl come from the previous cell
ja_train = [x for x, k in zip(ja_train, keep) if k]
en_train = [x for x, k in zip(en_train, keep) if k]
print(len(ja_train), len(en_train))

438269 438269


In [23]:
ja_val = [p["ja"] for p in val_kftt["translation"]]
en_val = [p["en"] for p in val_kftt["translation"]]

src_val = sp_ja.encode(ja_val, out_type=int, add_eos=True)
tgt_val = sp_en.encode(en_val, out_type=int, add_bos=True, add_eos=True)

In [24]:
class TrainDataset(Dataset):
    def __init__(self,ja_text,en_text,sp_ja,sp_en):
        self.ja = ja_text
        self.en = en_text
        self.sp_ja = sp_ja
        self.sp_en = sp_en
    def __len__(self):
        return len(self.ja)
    def __getitem__(self, idx):
        s = self.sp_ja.encode(self.ja[idx], out_type=int, add_eos=True)
        t = self.sp_en.encode(self.en[idx], out_type=int, add_bos=True, add_eos=True)
        return torch.tensor(s, dtype=torch.long), torch.tensor(t, dtype=torch.long)


In [25]:
train_dataset = TrainDataset(ja_train,en_train,sp_ja,sp_en)
ja,en = train_dataset[0]
ja,en

(tensor([  263,  1353,  2825,   265,   829,   821,  3211,   260,   464,   479,
           269,   265,  2225,   778,   269,   441,  5647,   313,   269,   265,
          2768,   288,   269,   264,   264,   266,   474,   273,   260,   417,
          4440,  6625,  3041, 11261,   301,   271,  8356,   273,   260,   742,
           977,   473, 10014,   839,   261,     3]),
 tensor([    2,  2843,  3554,   350,   275, 11699,  5480,  2158,   353,   531,
          3120,   292,   280,   271,   301,  3570,  3725,   264,   781,  1320,
          1886,   266,   260,   723,   305,   266,   260,  1350,   827,   262,
           260,   531,   372,   696,   261,   264,   271,   323,   269,   955,
          3725,   263,     3]))

In [26]:
class CustomDataset(Dataset):
    def __init__(self,src,tgt):
        self.src = src
        self.tgt = tgt
    def __len__(self):
        return len(self.src)
    def __getitem__(self,i):
        return torch.tensor(self.src[i], dtype=torch.long), torch.tensor(self.tgt[i], dtype=torch.long)
    
    
    

In [27]:
val_dataset = CustomDataset(src_val,tgt_val)
ja, en = val_dataset[0]
ja, en

(tensor([  263,  4374,   265,  2568, 14208,   393,   260,  8121,  6887,  3211,
           264,   266,   260,   674,  1384,   488,   301,   628,   393,   265,
           893,   768,   280,   539,  3211,   264,   265,  2568,  2363,   260,
         15159,  5908,   393,   260,  7790,   260,  1042,   462,   393,   260,
         10946,   393,   264,  2346,   273,   260,   644,   262,  2568,  2363,
           344,  1167,   265,  6203,   312,  2219,   269,   264,   270,  7917,
           449,   261,     3]),
 tensor([   2, 4423,  781,  641,  273,  331,  262,  260,  566,  839, 1055,  267,
          509, 6231,  625,  267,  262,  781,  265, 1924,  321, 2093,  261,  402,
          803,  468,  261, 5251,  261, 2239,  480,  261,  645, 1372,  272,  264,
         1464,  721, 5049, 6567, 7399,  455,  265, 2983,  353,  617, 2849,  272,
          262, 1157,  271,  382, 1202,  263,    3]))

In [28]:
PAD = 0
def collate_fn(batch):
    s,t= zip(*batch)
    s = pad_sequence(s , batch_first = True,padding_value = PAD)
    t = pad_sequence(t , batch_first = True,padding_value = PAD)
    return s , t[: ,:-1],t[:,1:]
    

In [29]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers = 2,
    pin_memory = True,
    persistent_workers = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers = 2,
    pin_memory = True,
    persistent_workers = True
)

In [30]:
s, target, labels = next(iter(train_loader))

print(s.shape, target.shape, labels.shape)
print(s[0])
print(target[0])
print(labels[0])

torch.Size([64, 66]) torch.Size([64, 114]) torch.Size([64, 114])
tensor([ 263, 4577,  266, 1344,  976, 1433,  261,    3,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0])
tensor([   2, 4694,  325,  545,  270,  340,  421,  271,  260,  753,  262,  269,
         823,  265, 5409,  506,  293,    3,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          

In [31]:
import math
import torch
import torch.nn as nn

PAD, BOS, EOS = 0, 2, 3

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, nhead=4,
                 enc_layers=3, dec_layers=3, ff=1024, dropout=0.1, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=PAD)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=PAD)
        self.pos = PositionalEncoding(d_model, max_len, dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=enc_layers, num_decoder_layers=dec_layers,
            dim_feedforward=ff, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.out = nn.Linear(d_model, tgt_vocab, bias=False)
        self.out.weight = self.tgt_emb.weight          # tie decoder input/output embeddings

        # init
        for p in self.transformer.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        for emb in (self.src_emb, self.tgt_emb):
            nn.init.normal_(emb.weight, mean=0.0, std=d_model ** -0.5)
            with torch.no_grad():
                emb.weight[PAD].zero_()

    def encode(self, src):
        src_pad = src == PAD
        x = self.pos(self.src_emb(src) * math.sqrt(self.d_model))
        memory = self.transformer.encoder(x, src_key_padding_mask=src_pad)
        return memory, src_pad

    def decode(self, tgt_in, memory, src_pad):
        T = tgt_in.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt_in.device), 1)
        y = self.pos(self.tgt_emb(tgt_in) * math.sqrt(self.d_model))
        h = self.transformer.decoder(
            y, memory,
            tgt_mask=causal,
            tgt_key_padding_mask=(tgt_in == PAD),
            memory_key_padding_mask=src_pad,
        )
        return self.out(h)                              # (batch, T, tgt_vocab)

    def forward(self, src, tgt_in):
        memory, src_pad = self.encode(src)
        return self.decode(tgt_in, memory, src_pad)

In [34]:
import math, time, os, torch, torch.nn as nn, sacrebleu
from torch.nn.utils.rnn import pad_sequence

PAD, BOS, EOS = 0, 2, 3
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

model = Seq2SeqTransformer(
    src_vocab=sp_ja.get_piece_size(), tgt_vocab=sp_en.get_piece_size(),
    d_model=512, nhead=8, enc_layers=4, dec_layers=4, ff=2048, dropout=0.2,
).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)
val_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

PEAK_LR, WARMUP = 5e-4, 4000
decay = [p for p in model.parameters() if p.dim() >= 2]
no_decay = [p for p in model.parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 0.01}, {"params": no_decay, "weight_decay": 0.0}],
    lr=PEAK_LR, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda step: min((step + 1) / WARMUP, math.sqrt(WARMUP / (step + 1))))
scaler = torch.amp.GradScaler("cuda")

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


In [35]:
def ids_to_text(ids):
    out = []
    for i in ids:
        if i == EOS: break
        if i not in (PAD, BOS): out.append(i)
    return sp_en.decode(out)

def make_batches(src_list, bs):
    order = sorted(range(len(src_list)), key=lambda i: len(src_list[i]))   # similar lengths together
    for i in range(0, len(order), bs):
        ids = order[i:i + bs]
        batch = pad_sequence([torch.tensor(src_list[j]) for j in ids],
                             batch_first=True, padding_value=PAD).to(device)
        yield ids, batch

@torch.no_grad()
def greedy_decode(src):
    max_len = min(int(src.size(1) * 1.5) + 10, 300)       # relative to the source, not fixed at 128
    with torch.autocast("cuda", dtype=torch.float16):
        memory, src_pad = model.encode(src)
        ys = torch.full((src.size(0), 1), BOS, dtype=torch.long, device=src.device)
        done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
        for _ in range(max_len):
            nxt = model.decode(ys, memory, src_pad)[:, -1].argmax(-1).masked_fill(done, PAD)
            ys = torch.cat([ys, nxt[:, None]], 1)
            done |= nxt == EOS
            if done.all(): break
    return ys.tolist()

def translate(src_list, decode_fn, bs=64, **kw):
    model.eval()
    out = [None] * len(src_list)
    for ids, src in make_batches(src_list, bs):
        for j, h in zip(ids, decode_fn(src, **kw)):
            out[j] = ids_to_text(h)
    return out

In [36]:
EPOCHS, PATIENCE, KEEP = 20, 4, 5
RESUME = False                         # set True only to continue after a crash

val_refs = [clean(x) for x in en_val]
val_src_eval = [x if len(x) <= 512 else x[:511] + [EOS] for x in src_val]

start_epoch, best_bleu, bad, ckpts = 1, -1.0, 0, []      # ckpts: (val_bleu, epoch, path)
if RESUME and os.path.exists("last.pt"):
    st = torch.load("last.pt", map_location=device, weights_only=False)
    model.load_state_dict(st["model"]); optimizer.load_state_dict(st["opt"])
    scheduler.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
    start_epoch, best_bleu, bad, ckpts = st["epoch"] + 1, st["best_bleu"], st["bad"], st["ckpts"]

print("params", sum(p.numel() for p in model.parameters()) / 1e6, "M")

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_total, tr_n = torch.zeros((), device=device), 0
    for s, dec_in, labels in train_loader:
        s, dec_in, labels = s.to(device, non_blocking=True), dec_in.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(s, dec_in)
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step()
        tr_total += loss.detach(); tr_n += 1          # no per-step GPU sync

    model.eval()
    va_total, va_n = torch.zeros((), device=device), 0
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for s, dec_in, labels in val_loader:
            s, dec_in, labels = s.to(device), dec_in.to(device), labels.to(device)
            logits = model(s, dec_in)
            va_total += val_loss_fn(logits.reshape(-1, logits.size(-1)), labels.reshape(-1)).detach(); va_n += 1
    tr, va = (tr_total / tr_n).item(), (va_total / va_n).item()

    val_bleu = sacrebleu.corpus_bleu(translate(val_src_eval, greedy_decode), [val_refs]).score
    print(f"epoch {epoch} | train {tr:.3f} | val {va:.3f} | val BLEU {val_bleu:.2f} | {(time.time()-t0)/60:.1f} min")

    path = f"ckpt_ep{epoch}.pt"                        # keep the KEEP best epochs by BLEU
    torch.save(model.state_dict(), path)
    ckpts.append((val_bleu, epoch, path))
    if len(ckpts) > KEEP:
        ckpts.sort(reverse=True)
        os.remove(ckpts.pop()[2])

    if val_bleu > best_bleu: best_bleu, bad = val_bleu, 0
    else: bad += 1

    torch.save({"model": model.state_dict(), "opt": optimizer.state_dict(),
                "sched": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "epoch": epoch, "best_bleu": best_bleu, "bad": bad, "ckpts": ckpts}, "last.pt")
    if bad >= PATIENCE:
        print("early stopping"); break

print("best val BLEU", best_bleu)

params 45.811712 M
epoch 1 | train 5.521 | val 3.642 | val BLEU 7.55 | 19.2 min
epoch 2 | train 4.243 | val 2.987 | val BLEU 12.80 | 19.3 min
epoch 3 | train 3.865 | val 2.714 | val BLEU 15.27 | 19.2 min
epoch 4 | train 3.678 | val 2.585 | val BLEU 16.48 | 19.2 min
epoch 5 | train 3.562 | val 2.512 | val BLEU 17.38 | 19.3 min
epoch 6 | train 3.481 | val 2.446 | val BLEU 17.41 | 19.3 min
epoch 7 | train 3.420 | val 2.402 | val BLEU 18.58 | 19.2 min
epoch 8 | train 3.372 | val 2.367 | val BLEU 19.00 | 19.2 min
epoch 9 | train 3.333 | val 2.330 | val BLEU 18.73 | 19.2 min
epoch 10 | train 3.299 | val 2.321 | val BLEU 18.93 | 19.2 min


KeyboardInterrupt: 

In [37]:
@torch.no_grad()
def beam_decode(src, beam=5, alpha=0.6):
    B, K, dev = src.size(0), beam, src.device
    max_len = min(int(src.size(1) * 1.5) + 10, 300)
    with torch.autocast("cuda", dtype=torch.float16):
        memory, src_pad = model.encode(src)
        memory, src_pad = memory.repeat_interleave(K, 0), src_pad.repeat_interleave(K, 0)
        ys = torch.full((B * K, 1), BOS, dtype=torch.long, device=dev)
        scores = torch.zeros(B, K, device=dev); scores[:, 1:] = -1e9   # one live beam at step 1
        finals = [[] for _ in range(B)]                                # (normalised score, ids)
        for step in range(1, max_len + 1):
            logp = torch.log_softmax(model.decode(ys, memory, src_pad)[:, -1].float(), -1).view(B, K, -1)
            V = logp.size(-1)
            top_s, top_i = (scores.unsqueeze(-1) + logp).view(B, K * V).topk(2 * K, dim=-1)
            b_idx, tok = top_i // V, top_i % V
            is_eos = tok == EOS
            norm = ((5.0 + step) / 6.0) ** alpha                       # length penalty (Wu et al.)
            for b, r in is_eos[:, :K].nonzero().tolist():              # finished hypotheses
                if len(finals[b]) < K:
                    finals[b].append((top_s[b, r].item() / norm, ys[b * K + b_idx[b, r].item(), 1:].tolist()))
            if all(len(f) >= K for f in finals): break
            order = torch.sort(is_eos.int(), dim=1, stable=True).indices[:, :K]   # K best non-EOS
            scores = top_s.gather(1, order)
            sel_beam, sel_tok = b_idx.gather(1, order), tok.gather(1, order)
            ys = ys.view(B, K, -1)
            ys = ys.gather(1, sel_beam.unsqueeze(-1).expand(-1, -1, ys.size(-1)))
            ys = torch.cat([ys, sel_tok.unsqueeze(-1)], -1).view(B * K, -1)
    ys = ys.view(B, K, -1)
    return [max(f, key=lambda x: x[0])[1] if f else ys[b, 0, 1:].tolist() for b, f in enumerate(finals)]

In [38]:
ckpts.sort(reverse=True)
print("kept epochs:", [(round(b, 2), e) for b, e, _ in ckpts])

avg = None
for _, _, p in ckpts:
    st = torch.load(p, map_location="cpu")
    if avg is None: avg = {k: v.float().clone() for k, v in st.items()}
    else:
        for k in avg: avg[k] += st[k].float()
avg = {k: v / len(ckpts) for k, v in avg.items()}
single = torch.load(ckpts[0][2], map_location="cpu")

def val_bleu_of(state):
    model.load_state_dict(state)
    return sacrebleu.corpus_bleu(translate(val_src_eval, greedy_decode), [val_refs]).score

b_single, b_avg = val_bleu_of(single), val_bleu_of(avg)
print("best single:", b_single, "| averaged:", b_avg)
model.load_state_dict(avg if b_avg >= b_single else single)

kept epochs: [(19.0, 8), (18.93, 10), (18.73, 9), (18.58, 7), (17.41, 6)]
best single: 19.00423908276733 | averaged: 18.66333316564693


<All keys matched successfully>

In [39]:
for alpha in (0.4, 0.6, 0.8, 1.0):
    hyps = translate(val_src_eval, beam_decode, bs=32, beam=5, alpha=alpha)
    print(alpha, sacrebleu.corpus_bleu(hyps, [val_refs]).score)

0.4 19.086688675664305
0.6 19.205879125836958
0.8 19.280249199341743
1.0 19.425482215174576


In [40]:
BEST_ALPHA = 1.0    # replace with the winner from above
test_src = [x if len(x) <= 512 else x[:511] + [EOS] for x in sp_ja.encode([clean(p["ja"]) for p in test_kftt["translation"]], add_eos=True)]
test_refs = [clean(p["en"]) for p in test_kftt["translation"]]

g = sacrebleu.corpus_bleu(translate(test_src, greedy_decode), [test_refs])
hyps = translate(test_src, beam_decode, bs=32, beam=5, alpha=BEST_ALPHA)
b = sacrebleu.corpus_bleu(hyps, [test_refs])
print("greedy:", g); print("beam:  ", b)
for i in range(5): print("REF:", test_refs[i]); print("HYP:", hyps[i]); print()

greedy: BLEU = 20.54 52.9/25.9/15.1/9.6 (BP = 0.973 ratio = 0.973 hyp_len = 25843 ref_len = 26560)
beam:   BLEU = 21.49 56.3/29.2/17.6/11.3 (BP = 0.900 ratio = 0.904 hyp_len = 24022 ref_len = 26560)
REF: Infobox Buddhist
HYP: INPON

REF: Dogen was a Zen monk in the early Kamakura period.
HYP: Dogen was a Zen priest in the early Kamakura period.

REF: The founder of Soto Zen
HYP: He was the founder of the Soto sect.

REF: Later in his life he also went by the name Kigen.
HYP: He was also called Kigen in his later years.

REF: Within the sect he is referred to by the honorary title Koso.
HYP: In the same sect, he was called Koso.

